# Tool-Calling Agent

## Goal

Learn tool calling directly, then build the agent loop.

Core flow:

`LLM → Tool Call → Application executes → ToolMessage → LLM → Final Answer`

**Tool call = request. Tool execution = your application.**

In [ ]:
%pip install -q -U langchain-core langchain-openai

In [ ]:
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"]=getpass("OPENAI_API_KEY: ")

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

llm=ChatOpenAI(model="gpt-5-mini",temperature=0)

## 1. Two tools

In [ ]:
@tool
def calculator(expression: str) -> str:
    allowed=set("0123456789+-*/(). %")
    if not set(expression)<=allowed:
        return "Error: unsupported characters."
    try:
        return str(eval(expression,{"__builtins__":{}},{}))
    except Exception as e:
        return f"Error: {e}"

@tool
def temperature_convert(celsius: float) -> str:
    return f"{(celsius*9/5)+32:.2f} °F"

tools=[calculator,temperature_convert]
tool_map={t.name:t for t in tools}

for t in tools:
    print(t.name)
    print(t.description)
    print(t.args_schema.model_json_schema())

## 2. Bind and inspect

The model can return structured tool calls.

In [ ]:
model=llm.bind_tools(tools)

response=model.invoke("Convert 25 Celsius to Fahrenheit.")
print(response.tool_calls)

for call in response.tool_calls:
    print("name:",call["name"])
    print("args:",call["args"])
    print("id:",call["id"])

## 3. Execute and return a ToolMessage

In [ ]:
messages=[HumanMessage(content="Convert 25 Celsius to Fahrenheit."),response]

for call in response.tool_calls:
    tool=tool_map[call["name"]]
    result=tool.invoke(call["args"])
    messages.append(ToolMessage(
        content=str(result),
        tool_call_id=call["id"]
    ))

final=model.invoke(messages)
print(final.content)

## 4. Multiple tools

The same pattern works for calculator questions.

In [ ]:
for query in [
    "Calculate (100 + 25) / 5.",
    "Convert 30 Celsius to Fahrenheit.",
]:
    first=model.invoke(query)
    messages=[HumanMessage(content=query),first]

    for call in first.tool_calls:
        result=tool_map[call["name"]].invoke(call["args"])
        messages.append(ToolMessage(
            content=str(result),
            tool_call_id=call["id"]
        ))

    final=model.invoke(messages)
    print("\nUSER:",query)
    print("ANSWER:",final.content)

## 5. Turn it into an agent

A tool-calling agent repeats the same mechanics until the model has no more tool calls.

In [ ]:
def tool_calling_agent(query,max_steps=5):
    messages=[HumanMessage(content=query)]

    for _ in range(max_steps):
        response=model.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            selected=tool_map.get(call["name"])
            try:
                result=selected.invoke(call["args"]) if selected else "Unknown tool"
            except Exception as e:
                result=f"Tool error: {e}"

            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            ))

    return "Stopped: maximum steps reached."

print(tool_calling_agent(
    "Calculate 125 * 8 and convert 25 Celsius to Fahrenheit."
))

## 6. Typed schemas

Complex tools should validate inputs.

Example: search with a bounded `max_results`.

In [ ]:
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query:str=Field(description="Search query")
    max_results:int=Field(default=5,ge=1,le=20)

print(SearchInput.model_json_schema())

## 7. Errors and safety

Tool failure should become useful information to the model.

Production controls:
- validation
- permissions
- authentication
- retries/timeouts
- step limits
- logging
- human approval for sensitive actions

## 8. Framework shortcut

After understanding the manual loop, current LangChain also provides `create_agent(model=..., tools=...)`.

The abstraction is useful. Understanding what it hides is even more useful.

In [ ]:
from langchain.agents import create_agent

agent=create_agent(
    model=llm,
    tools=[calculator,temperature_convert]
)

# Example:
# result=agent.invoke({
#     "messages":[
#         {"role":"user","content":"Calculate 17 * 24."}
#     ]
# })
# print(result)

## 9. Evaluation

For tool-calling agents, measure:
- tool selection
- argument correctness
- call count/order
- error handling
- final answer
- stopping behavior

This is the agent equivalent of testing retrieval quality in RAG.

In [ ]:
tests=[
    ("Calculate 10 * 20.","calculator"),
    ("Convert 20 Celsius to Fahrenheit.","temperature_convert"),
]

for query,expected in tests:
    first=model.invoke(query)
    chosen=first.tool_calls[0]["name"] if first.tool_calls else None
    print("PASS" if chosen==expected else "CHECK",expected,chosen)

## Final mental model

Tool calling:
`User → LLM → Tool Call → Tool → ToolMessage → LLM → Answer`

Agent:
`Goal → Decide → Act → Observe → Decide again → ... → Answer`

Remember:
1. Tools extend an LLM.
2. Tools need clear descriptions/schemas.
3. Binding exposes tools.
4. A tool call is not execution.
5. Your app executes and returns the result.
6. The loop is what turns tool calling into an agent.